# Spark publishes to Kafka

Three ways Spark writes to Kafka in this lab:

* `kafka-enrich-publish`: stream joins events with Postgres `customers`, publishes
  `orders.enriched` and `orders.alerts`
* `kafka-windowed-metrics`: event-time windows with a watermark, publishes `orders.metrics`
  (compacted topic) and upserts Postgres `order_metrics_live`
* `kafka-batch-publish`: a plain batch job backfills `orders.backfill` from Hive `demo.orders`

Then `verify-streaming` checks every output against what the producer sent.

In [ ]:
import kyuubi_lab as lab          # client/kyuubi_lab.py (on PYTHONPATH in the Jupyter image)
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
RID = lab.new_run_id()            # one id for every job in this notebook run
print("Kyuubi REST", lab.REST_URL, "| JDBC", f"{lab.JDBC_HOST}:{lab.JDBC_PORT}", "| user", lab.USER, "| run", RID)

In [ ]:
import lab_runner
assert lab_runner.kafka_reachable(), "Kafka isn't running - make kafka-install"
lab.produce(orders=30, run_id=RID)

In [ ]:
lab.run('kafka-enrich-publish', run_id=RID)
enr = lab.topic('orders.enriched', last=300, run_id=RID)
enr.head()

In [ ]:
enr.groupby('tier').size().plot.pie(autopct='%1.0f%%', title='events by customer tier') if 'tier' in enr else enr.columns

In [ ]:
lab.topic('orders.alerts', last=20)

In [ ]:
lab.run('kafka-windowed-metrics', run_id=RID)
lab.topic('orders.metrics', last=20)

In [ ]:
lab.pg("SELECT * FROM order_metrics_live ORDER BY window_start DESC, city LIMIT 10")

In [ ]:
lab.run('kafka-batch-publish', run_id=RID)
lab.topic('orders.backfill', last=5)

## Everything agrees?
`verify-streaming` needs the file-event stream of the same run too, so drop files and run it first.

In [ ]:
lab.produce(orders=20, run_id=RID + "f", sink="s3", files=3)
lab.run('file-drop-stream', run_id=RID)
lab.run('kafka-consumer-basic', run_id=RID); lab.run('kafka-to-delta-pg', run_id=RID)
r = lab.run('verify-streaming', run_id=RID)
lab.log_tail(r, 'OK')